# Test i Vizualizimeve

Test i **3 grafikëve të rinj** para integrimit në `src/`:
- **Figura 3** — Scatter Plot: GDP per Capita vs Vaksinimi
- **Figura 4** — Histogram i CFR me vijë mesatare dhe mediane
- **Figura 5** — K-Means Clustering (3 grupe)

## 1. Ngarkimi i Bibliotekave dhe Dataset-it

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.figsize':   (12, 7),
    'figure.dpi':       120,
    'font.size':        11,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
    'axes.labelsize':   12,
})

df = pd.read_csv('../data/processed/covid_clean.csv', parse_dates=['date'])
print(f'Dataset: {df.shape[0]} vende, {df.shape[1]} kolona')
df[['location', 'CFR', 'Cases_per_100k', 'gdp_per_capita',
    'total_vaccinations_per_hundred', 'total_cases']].head()

---
## Figura 3 — Scatter Plot: GDP per Capita vs Vaksinimi

- Boshti X: `gdp_per_capita`
- Boshti Y: `total_vaccinations_per_hundred`
- Madhësia e pikave proporcionale me `total_cases`
- Vija e trendit me `np.polyfit()`
- Etiketat e vendeve

In [ ]:
x = df['gdp_per_capita']
y = df['total_vaccinations_per_hundred']

# Madhësia e pikave: normalizim ndërmjet 60 dhe 600
sizes = (df['total_cases'] / df['total_cases'].max()) * 540 + 60

fig, ax = plt.subplots()
ax.scatter(x, y, s=sizes, color='#4C72B0', alpha=0.7,
           edgecolors='white', linewidths=0.8)

# Vija e trendit
coeffs = np.polyfit(x, y, deg=1)
x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, np.polyval(coeffs, x_line),
        color='#C44E52', linewidth=1.8, linestyle='--', label='Trend linear')

# Etiketat e vendeve
for _, row in df.iterrows():
    ax.annotate(row['location'],
                xy=(row['gdp_per_capita'], row['total_vaccinations_per_hundred']),
                xytext=(4, 4), textcoords='offset points', fontsize=7.5)

ax.set_xlabel('GDP per Capita (USD)')
ax.set_ylabel('Vaksinime për 100 Banorë')
ax.set_title('Figura 3: GDP per Capita vs Vaksinimi — 20 Vende Evropiane')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Figura 4 — Histogram i CFR

- 10 bins
- Vijë e kuqe (e ndërprerë) për **mesataren**
- Vijë jeshile (e plotë) për **medianën**

In [ ]:
cfr        = df['CFR'].dropna()
mean_val   = cfr.mean()
median_val = cfr.median()

print(f'Mesatare CFR : {mean_val:.2f}%')
print(f'Mediana  CFR : {median_val:.2f}%')

fig, ax = plt.subplots()
ax.hist(cfr, bins=10, color='#4C72B0', edgecolor='white', alpha=0.85)

ax.axvline(mean_val,   color='#C44E52', linewidth=2,
           linestyle='--', label=f'Mesatare: {mean_val:.2f}%')
ax.axvline(median_val, color='#55A868', linewidth=2,
           linestyle='-',  label=f'Mediana:  {median_val:.2f}%')

ax.set_xlabel('Case Fatality Rate — CFR (%)')
ax.set_ylabel('Numri i Vendeve')
ax.set_title('Figura 4: Shpërndarja e CFR — 20 Vende Evropiane')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## Figura 5 — K-Means Clustering (3 Grupe)

- Ngjyra të ndryshme për secilin cluster
- Etiketat e vendeve pranë çdo pike
- Boshtet: `Cases_per_100k` vs `CFR`

In [ ]:
CLUSTER_FEATURES = ['Cases_per_100k', 'CFR',
                    'total_vaccinations_per_hundred', 'gdp_per_capita']
COLORS = ['#4C72B0', '#C44E52', '#55A868']

# K-Means
df_vis = df.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_vis[CLUSTER_FEATURES])
df_vis['Cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

# Plot
fig, ax = plt.subplots()
for c in sorted(df_vis['Cluster'].unique()):
    sub = df_vis[df_vis['Cluster'] == c]
    ax.scatter(sub['Cases_per_100k'], sub['CFR'],
               color=COLORS[c], s=120, edgecolors='white',
               linewidths=0.8, label=f'Cluster {c}', zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['location'],
                    xy=(row['Cases_per_100k'], row['CFR']),
                    xytext=(5, 4), textcoords='offset points',
                    fontsize=7.5, color=COLORS[c])

ax.set_xlabel('Rastet për 100,000 Banorë')
ax.set_ylabel('CFR (%)')
ax.set_title('Figura 5: K-Means Clustering — 20 Vende Evropiane (3 Grupe)')
ax.legend(title='Cluster', fontsize=10)
plt.tight_layout()
plt.show()

---
## Konfirmim: Vendet për Çdo Cluster

In [ ]:
for c in sorted(df_vis['Cluster'].unique()):
    vendet = df_vis[df_vis['Cluster'] == c]['location'].tolist()
    print(f'Cluster {c}: {vendet}')